# SimCLR Ablation Study — Session B: Pairwise Augmentation Experiments
Only 2 augmentations active at a time.

| Experiment | Augmentations active |
|---|---|
| `pair_crop_jitter` | crop + color jitter |
| `pair_crop_grayscale` | crop + grayscale |
| `pair_crop_blur` | crop + blur |
| `pair_jitter_grayscale` | color jitter + grayscale |

**Expected runtime:** ~1.5h × 4 = ~6h on T4. Fits in one 12h Kaggle session.  
**Prerequisites:** Repo pushed to GitHub with all configs.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'No GPU — enable GPU in Kaggle settings.'

In [ ]:
import os, subprocess

REPO_URL   = 'https://github.com/SAlaMusa/IDL.git'
REPO_DIR   = '/kaggle/working/IDL'
OUTPUT_DIR = '/kaggle/working'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import sys, glob, shutil, csv, datetime
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
print('Imports OK')

In [ ]:
RESULTS_CSV = os.path.join(OUTPUT_DIR, 'session_b_results.csv')

if not os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['exp_name', 'config', 'best_top1', 'checkpoint', 'timestamp'])


def run_experiment(exp_name, config_path, epochs=200, batch=256):
    ckpt_name = f'{exp_name}_ep{epochs}.pth.tar'
    ckpt_path = os.path.join(OUTPUT_DIR, ckpt_name)

    if os.path.exists(ckpt_path):
        print(f'[{exp_name}] Checkpoint exists, skipping pretraining.')
    else:
        print(f"\n{'='*60}\n[{exp_name}] PRETRAINING\n{'='*60}")
        result = subprocess.run([
            'python', 'run.py',
            '--config', config_path,
            '--epochs',     str(epochs),
            '--batch-size', str(batch),
            '-j', '2', '--seed', '42',
            '--fp16-precision',
        ])
        if result.returncode != 0:
            raise RuntimeError(f'Pretraining failed for {exp_name}.')
        latest = max(glob.glob('runs/**/*.pth.tar', recursive=True), key=os.path.getmtime)
        shutil.copy2(latest, ckpt_path)
        print(f'[{exp_name}] Checkpoint saved: {ckpt_path}')

    print(f'\n[{exp_name}] LINEAR EVALUATION')
    eval_result = subprocess.run([
        'python', 'linear_eval.py',
        '--checkpoint', ckpt_path,
        '--dataset',    'cifar10',
        '--arch',       'resnet18',
        '--epochs',     '100',
        '-b',           '256',
        '-j',           '2',
        '--seed',       '42',
    ])
    if eval_result.returncode != 0:
        raise RuntimeError(f'Linear eval failed for {exp_name}.')

    with open('linear_eval_results.csv') as f:
        best_top1 = float(list(csv.DictReader(f))[-1]['best_top1'])

    with open(RESULTS_CSV, 'a', newline='') as f:
        csv.writer(f).writerow([
            exp_name, config_path, f'{best_top1:.2f}', ckpt_path,
            datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
        ])
    print(f'[{exp_name}] Best Top-1: {best_top1:.2f}%')
    return best_top1


print('Helper defined.')

## Run Pairwise Experiments

In [ ]:
pairs = [
    ('pair_crop_jitter',      'configs/pair_crop_jitter.yaml'),
    ('pair_crop_grayscale',   'configs/pair_crop_grayscale.yaml'),
    ('pair_crop_blur',        'configs/pair_crop_blur.yaml'),
    ('pair_jitter_grayscale', 'configs/pair_jitter_grayscale.yaml'),
]

results = {}
for exp_name, config_path in pairs:
    results[exp_name] = run_experiment(exp_name, config_path)

print('\n=== Session B Complete ===')
for name, acc in results.items():
    print(f'  {name:<30} {acc:.2f}%')
print(f'\nResults saved to: {RESULTS_CSV}')